# IPPO (PettingZoo MPE) - Colab Notebook

This notebook combines the current `env.py`, `trainer.py`, `ppoagent.py`, `rolloutBuffer.py`, and `networks.py` logic into one runnable Colab flow.

1. In Colab, set runtime to **GPU**.
2. Run cells top-to-bottom.
3. Training runs without window rendering; policy playback is rendered inline as a GIF.


In [6]:
!pip -q install pettingzoo[mpe] pygame matplotlib imageio


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from IPython.display import Image, display

from pettingzoo.mpe import simple_spread_v3


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


class ActorNetwork(nn.Module):
    def __init__(self, obs_dim, hidden_dim, actions_dim, device=None):
        super(ActorNetwork, self).__init__()
        self.device = torch.device(device) if device is not None else get_device()

        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, actions_dim)

        self._init_weights()
        self.to(self.device)

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc2.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc3.weight, gain=0.01)

        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.constant_(self.fc2.bias, 0.0)
        nn.init.constant_(self.fc3.bias, 0.0)

    def forward(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)

        x = self.fc1(obs)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x

    def get_action_and_log_probs(self, obs, action=None):
        logits = self.forward(obs)
        dist = Categorical(logits=logits)

        if action is None:
            action = dist.sample()
        else:
            if not torch.is_tensor(action):
                action = torch.as_tensor(action, dtype=torch.long, device=self.device)
            else:
                action = action.to(self.device, dtype=torch.long)

        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
        return action, log_prob, entropy

    def evaluate_actions(self, obs, actions):
        if not torch.is_tensor(actions):
            actions = torch.as_tensor(actions, dtype=torch.long, device=self.device)
        else:
            actions = actions.to(self.device, dtype=torch.long)

        logits = self.forward(obs)
        dist = Categorical(logits=logits)
        log_prob = dist.log_prob(actions)
        entropy = dist.entropy()
        return log_prob, entropy, logits


class CriticNetwork(nn.Module):
    def __init__(self, obs_dims, hidden_dims, device=None):
        super(CriticNetwork, self).__init__()
        self.device = torch.device(device) if device is not None else get_device()
        
        self.fc1 = nn.Linear(obs_dims, hidden_dims)
        self.fc2 = nn.Linear(hidden_dims, hidden_dims)
        self.fc3 = nn.Linear(hidden_dims, 1)

        self._init_weights()
        self.to(self.device)

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc2.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc3.weight, gain=1.0)

        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.constant_(self.fc2.bias, 0.0)
        nn.init.constant_(self.fc3.bias, 0.0)

    def forward(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)

        x = self.fc1(obs)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x


class RolloutBuffer:
    def __init__(self, buffer_size, obs_dim, action_dim, gamma, gae_lambda, device):
        self.buffer_size = buffer_size
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.device = device

        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.idx = 0

    def add_rollout(self, obs, action, reward, done, log_prob, value):
        self.obs.append(obs)
        self.actions.append(action)

        self.rewards.append(reward.item() if torch.is_tensor(reward) else reward)
        self.dones.append(done.item() if torch.is_tensor(done) else done)
        self.log_probs.append(log_prob.item() if torch.is_tensor(log_prob) else log_prob)
        self.values.append(value.item() if torch.is_tensor(value) else value)
        self.idx += 1

    def compute_returns_and_advantages(self, last_value):
        rewards = torch.as_tensor(self.rewards, dtype=torch.float32, device=self.device)
        values = torch.as_tensor(self.values, dtype=torch.float32, device=self.device)
        dones = torch.as_tensor(self.dones, dtype=torch.float32, device=self.device)

        advantages = torch.zeros(self.buffer_size, dtype=torch.float32, device=self.device)
        last_gae = 0

        for t in reversed(range(self.buffer_size)):
            if t == self.buffer_size - 1:
                if not torch.is_tensor(last_value):
                    next_value = torch.tensor(last_value, dtype=torch.float32, device=self.device)
                else:
                    next_value = last_value.to(self.device, dtype=torch.float32)
            else:
                next_value = values[t + 1]

            delta = rewards[t] + self.gamma * (1 - dones[t]) * next_value - values[t]
            advantages[t] = delta + self.gamma * self.gae_lambda * (1 - dones[t]) * last_gae
            last_gae = advantages[t]

        returns = advantages + values
        self.advantages = advantages
        self.returns = returns

    def get(self):
        obs = torch.as_tensor(np.asarray(self.obs), dtype=torch.float32, device=self.device)
        actions = torch.as_tensor(np.asarray(self.actions), dtype=torch.long, device=self.device)
        log_probs = torch.as_tensor(self.log_probs, dtype=torch.float32, device=self.device)

        adv = self.advantages
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        return obs, actions, log_probs, adv, self.returns

    def clear(self):
        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.idx = 0


class PPOAgent:
    def __init__(
        self,
        obs_dim,
        hidden_dim,
        action_dim,
        lr=3e-4,
        buffer_size=2048,
        gamma=0.99,
        gae_lambda=0.95,
        clip_epsilon=0.2,
        value_coef=0.5,
        entropy_coef=0.01,
        max_grad_norm=0.5,
    ):
        self.device = get_device()

        self.actor = ActorNetwork(obs_dim, hidden_dim, action_dim, device=self.device).to(self.device)
        self.critic = CriticNetwork(obs_dim, hidden_dim, device=self.device).to(self.device)
        self.buffer = RolloutBuffer(buffer_size, obs_dim, action_dim, gamma, gae_lambda, self.device)

        self.actor_optim = torch.optim.Adam(self.actor.parameters(), lr=lr)
        self.critic_optim = torch.optim.Adam(self.critic.parameters(), lr=lr)

        self.clip_eps = clip_epsilon
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        self.gamma = gamma

    def select_action(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)

        with torch.no_grad():
            actions, log_probs, _ = self.actor.get_action_and_log_probs(obs)
            value = self.critic.forward(obs)

        return actions.item(), log_probs.item(), value.squeeze(0).item()

    def update(self, last_obs, num_epochs=30, mini_batch_size=None):
        with torch.no_grad():
            last_obs_tensor = torch.as_tensor(last_obs, dtype=torch.float32, device=self.device).unsqueeze(0)
            last_value = self.critic(last_obs_tensor).squeeze()

        self.buffer.compute_returns_and_advantages(last_value)
        obs, actions, old_log_probs, advantages, returns = self.buffer.get()

        total_policy_loss = 0
        total_value_loss = 0
        total_entropy = 0

        for _ in range(num_epochs):
            new_log_probs, entropy, _ = self.actor.evaluate_actions(obs, actions)

            ratio = torch.exp(new_log_probs - old_log_probs)
            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1.0 - self.clip_eps, 1 + self.clip_eps) * advantages
            policy_loss = -torch.min(surr1, surr2).mean()

            values = self.critic(obs).squeeze(-1)
            values_loss = F.mse_loss(values, returns)

            entropy_loss = -entropy.mean()
            loss = policy_loss + self.value_coef * values_loss + self.entropy_coef * entropy_loss

            self.actor_optim.zero_grad()
            self.critic_optim.zero_grad()
            loss.backward()

            nn.utils.clip_grad_norm_(self.actor.parameters(), self.max_grad_norm)
            nn.utils.clip_grad_norm_(self.critic.parameters(), self.max_grad_norm)

            self.actor_optim.step()
            self.critic_optim.step()

            total_policy_loss += policy_loss.item()
            total_entropy += entropy.mean().item()
            total_value_loss += values_loss.item()

        with torch.no_grad():
            values = self.critic(obs).squeeze()
            var_returns = returns.var()
            var_residual = (returns - values).var()
            explained_var = 1 - var_residual / (var_returns + 1e-8)

            rewards_tensor = torch.as_tensor(self.buffer.rewards, dtype=torch.float32, device=self.device)
            values_tensor = torch.as_tensor(self.buffer.values, dtype=torch.float32, device=self.device)
            dones_tensor = torch.as_tensor(self.buffer.dones, dtype=torch.float32, device=self.device)

            next_values = torch.zeros_like(values_tensor)
            next_values[:-1] = values_tensor[1:]
            next_values[-1] = last_value

            td_errors = rewards_tensor + self.gamma * (1 - dones_tensor) * next_values - values_tensor
            mean_bellman_error = td_errors.abs().mean().item()

        self.buffer.clear()

        return {
            "policy_loss": total_policy_loss / num_epochs,
            "value_loss": total_value_loss / num_epochs,
            "entropy": total_entropy / num_epochs,
            "explained_variance": explained_var.item(),
            "mean_bellman_error": mean_bellman_error,
        }


class IPPOTrainer:
    def __init__(self, env, num_agents, obs_dim, hidden_dim, action_dim):
        self.env = env
        self.agents = {}
        self.num_agents = num_agents

        for agent_id in env.possible_agents:
            self.agents[agent_id] = PPOAgent(obs_dim, hidden_dim, action_dim)

        self.metrics_history = {
            "timesteps": [],
            "mean_episode_return": [],
            "mean_episode_length": [],
            "policy_loss": [],
            "value_loss": [],
            "entropy": [],
            "explained_variance": [],
            "mean_bellman_error": [],
        }

    def collect_rollouts(self, num_steps, obs):
        if obs is None:
            obs, info = self.env.reset()
            if getattr(self.env, "render_mode", None) == "human":
                self.env.render()

        episode_returns = {a_id: 0.0 for a_id in self.agents.keys()}
        episode_lengths = {a_id: 0 for a_id in self.agents.keys()}
        completed_episodes = []

        for _ in range(num_steps):
            actions = {}
            values = {}
            log_probs = {}

            for a_id, agent in self.agents.items():
                action, log_prob, value = agent.select_action(obs[a_id])
                actions[a_id] = action
                values[a_id] = value
                log_probs[a_id] = log_prob

            next_obs, rewards, dones, truncs, info = self.env.step(actions)

            if getattr(self.env, "render_mode", None) == "human":
                self.env.render()

            for a_id, agent in self.agents.items():
                agent.buffer.add_rollout(
                    obs[a_id],
                    actions[a_id],
                    rewards[a_id],
                    dones[a_id],
                    log_probs[a_id],
                    values[a_id],
                )

                episode_returns[a_id] += rewards[a_id]
                episode_lengths[a_id] += 1

            if all(dones.values()) or all(truncs.values()):
                obs, info = self.env.reset()

                completed_episodes.append(
                    {
                        "returns": {a_id: episode_returns[a_id] for a_id in self.agents.keys()},
                        "lengths": {a_id: episode_lengths[a_id] for a_id in self.agents.keys()},
                        "mean_return": sum(episode_returns.values()) / len(episode_returns),
                        "mean_length": sum(episode_lengths.values()) / len(episode_lengths),
                    }
                )

                episode_returns = {a_id: 0.0 for a_id in self.agents.keys()}
                episode_lengths = {a_id: 0 for a_id in self.agents.keys()}
            else:
                obs = next_obs

        return obs, completed_episodes

    def train(self, total_timesteps, rollout_length):
        timesteps = 0
        obs = None

        while timesteps < total_timesteps:
            last_obs, completed_episodes = self.collect_rollouts(rollout_length, obs)
            timesteps += rollout_length

            if completed_episodes:
                mean_return = sum(ep["mean_return"] for ep in completed_episodes) / len(completed_episodes)
                mean_length = sum(ep["mean_length"] for ep in completed_episodes) / len(completed_episodes)

                self.metrics_history["timesteps"].append(timesteps)
                self.metrics_history["mean_episode_return"].append(mean_return)
                self.metrics_history["mean_episode_length"].append(mean_length)

            all_agent_metrics = []
            for a_id, agent in self.agents.items():
                metrics = agent.update(last_obs[a_id])
                all_agent_metrics.append(metrics)

            avg_metrics = {
                key: sum(m[key] for m in all_agent_metrics) / len(all_agent_metrics)
                for key in all_agent_metrics[0].keys()
            }

            for k in ["policy_loss", "value_loss", "entropy", "explained_variance", "mean_bellman_error"]:
                self.metrics_history[k].append(avg_metrics[k])

            if timesteps % (rollout_length * 10) == 0:
                print(
                    f".      Policy Loss: {avg_metrics['policy_loss']:.4f}"
                    f".       Value Loss: {avg_metrics['value_loss']:.4f}"
                    f"           Entropy: {avg_metrics['entropy']:.4f}"
                    f".    Explained Var: {avg_metrics['explained_variance']:.4f}"
                    f"\nMean Bellman Error: {avg_metrics['mean_bellman_error']:.4f}"
                )

            obs = last_obs

    def plot_metrics(self, save_path="training_metrics.png"):
        fig, axes = plt.subplots(3, 3, figsize=(15, 12))
        fig.suptitle("IPPO Training Metrics", fontsize=16)

        if self.metrics_history["mean_episode_return"]:
            axes[0, 0].plot(self.metrics_history["timesteps"], self.metrics_history["mean_episode_return"])
            axes[0, 0].set_title("Mean Episode Return")
            axes[0, 0].set_xlabel("Timesteps")
            axes[0, 0].set_ylabel("Return")
            axes[0, 0].grid(True)

        if self.metrics_history["mean_episode_length"]:
            axes[0, 1].plot(self.metrics_history["timesteps"], self.metrics_history["mean_episode_length"])
            axes[0, 1].set_title("Mean Episode Length")
            axes[0, 1].set_xlabel("Timesteps")
            axes[0, 1].set_ylabel("Steps")
            axes[0, 1].grid(True)

        update_steps = list(range(len(self.metrics_history["policy_loss"])))

        axes[0, 2].plot(update_steps, self.metrics_history["policy_loss"])
        axes[0, 2].set_title("Policy Loss")
        axes[0, 2].set_xlabel("Updates")
        axes[0, 2].set_ylabel("Loss")
        axes[0, 2].grid(True)

        axes[1, 0].plot(update_steps, self.metrics_history["value_loss"])
        axes[1, 0].set_title("Value Loss")
        axes[1, 0].set_xlabel("Updates")
        axes[1, 0].set_ylabel("Loss")
        axes[1, 0].grid(True)

        axes[1, 1].plot(update_steps, self.metrics_history["entropy"])
        axes[1, 1].set_title("Entropy")
        axes[1, 1].set_xlabel("Updates")
        axes[1, 1].set_ylabel("Entropy")
        axes[1, 1].grid(True)

        axes[2, 1].plot(update_steps, self.metrics_history["explained_variance"])
        axes[2, 1].set_title("Explained Variance")
        axes[2, 1].set_xlabel("Updates")
        axes[2, 1].set_ylabel("Explained Var")
        axes[2, 1].grid(True)

        axes[2, 2].plot(update_steps, self.metrics_history["mean_bellman_error"])
        axes[2, 2].set_title("Mean Bellman Error")
        axes[2, 2].set_xlabel("Updates")
        axes[2, 2].set_ylabel("TD Error")
        axes[2, 2].grid(True)

        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        plt.show()
        print(f"Metrics plot saved to {save_path}")


print(f"Using device: {get_device()}")


Using device: cpu


In [8]:
NUM_AGENTS = 5
MAX_CYCLES = 100
TOTAL_TIMESTEPS = 200_000
ROLLOUT_LENGTH = 2048

train_env = simple_spread_v3.parallel_env(N=NUM_AGENTS, max_cycles=MAX_CYCLES)
obs, info = train_env.reset(seed=42)
print("Observation shape:", obs["agent_0"].shape)
print("Using Device:", get_device())
trainer = IPPOTrainer(train_env, NUM_AGENTS, obs["agent_0"].shape[0], hidden_dim=64, action_dim=5)
trainer.train(total_timesteps=TOTAL_TIMESTEPS, rollout_length=ROLLOUT_LENGTH)
trainer.plot_metrics("ippo_training_colab.png")


Observation shape: (30,)
Using Device: cpu


KeyboardInterrupt: 

In [ ]:
def run_policy_video(trainer, num_agents=5, max_cycles=100, episodes=3, fps=12, out_path="ippo_policy.gif"):
    eval_env = simple_spread_v3.parallel_env(
        N=num_agents,
        max_cycles=max_cycles,
        render_mode="rgb_array",
    )

    obs, info = eval_env.reset(seed=123)
    frames = []
    returns = {a_id: 0.0 for a_id in eval_env.possible_agents}
    completed = 0

    while completed < episodes:
        actions = {}
        for a_id, agent in trainer.agents.items():
            action, _, _ = agent.select_action(obs[a_id])
            actions[a_id] = action

        obs, rewards, dones, truncs, info = eval_env.step(actions)
        frame = eval_env.render()
        if frame is not None:
            frames.append(frame)

        for a_id in returns:
            returns[a_id] += rewards[a_id]

        if all(dones.values()) or all(truncs.values()):
            completed += 1
            if completed < episodes:
                obs, info = eval_env.reset()

    eval_env.close()

    if len(frames) == 0:
        raise RuntimeError("No frames captured. Check render_mode and environment setup.")

    imageio.mimsave(out_path, frames, fps=fps)
    return out_path, returns


gif_path, eval_returns = run_policy_video(
    trainer,
    num_agents=NUM_AGENTS,
    max_cycles=1000,
    episodes=1,
    fps=12,
    out_path="ippo_policy.gif",
)

print("Eval returns:", eval_returns)
display(Image(filename=gif_path))


Eval returns: {'agent_0': -28663.95449875764, 'agent_1': -29318.454498757685, 'agent_2': -28582.454498757663, 'agent_3': -29562.954498757677, 'agent_4': -28978.95449875767}
